# BioGPT: LLM-Based BioScript Compiler Automation
 
**Author:** Ishan Gain  
**Date:** August 2026  
**Platform:** Kaggle (GPU T4 x2)
 
---
 
## Abstract
 
This notebook presents **BioGPT**, a fine-tuned large language model pipeline 
for automatically generating BioScript (`.bs`) code from natural language 
descriptions of biological laboratory protocols.
 
BioScript is a domain-specific language (DSL) used by the 
[lilott8/BioScript](https://github.com/lilott8/BioScript) ANTLR4-based compiler 
for automating digital microfluidic (DMF) protocols. Writing BioScript manually 
requires deep knowledge of both the biological protocol and the compiler syntax — 
a significant barrier for wet-lab scientists.
 
**Our contributions:**
1. **OpenBioSet** — A curated dataset of 175 biological protocols with paired 
   natural language descriptions and BioScript code
2. **Data augmentation pipeline** — Growing 174 training pairs to 556 examples 
   using LLM-based paraphrasing (Groq API)
3. **Multi-model fine-tuning** — QLoRA fine-tuning of 4 models for comparison:
   - Qwen2.5-Coder-3B-Instruct (3 epochs)
   - Qwen2.5-Coder-3B-Instruct (5 epochs)
   - Llama-3.2-3B-Instruct (5 epochs)
   - Qwen2.5-Coder-7B-Instruct (5 epochs)
4. **Evaluation framework** — Structural accuracy, BLEU-1, and coverage metrics
 
---
 
## Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Dataset Loading & Exploration](#2-dataset-loading--exploration)
3. [Data Augmentation](#3-data-augmentation)
4. [Model Training](#4-model-training)
5. [Evaluation](#5-evaluation)
6. [Results & Comparison](#6-results--comparison)
7. [Inference Demo](#7-inference-demo)

## 1. Environment Setup
 
We use [Unsloth](https://github.com/unslothai/unsloth) for 2x faster QLoRA 
fine-tuning on Kaggle's T4 GPU. All dependencies are installed fresh to ensure 
reproducibility.
 
**Hardware:** Kaggle GPU T4 x2 (2 × 16GB VRAM)  
**Key libraries:** unsloth, transformers, trl, peft, datasets, groq

In [1]:
# ============================================================
# CELL 1 — CODE: Install Dependencies
# ============================================================

import subprocess, os
 
# Fix torchvision compatibility
subprocess.run([
    "pip", "install", "-q", "--force-reinstall",
    "--no-deps", "--no-cache-dir",
    "torchvision==0.25.0"
], check=True)
 
# Install Unsloth (handles QLoRA without bitsandbytes issues)
subprocess.run(["pip", "install", "-q", "unsloth[kaggle-new]"], check=True)
 
# Install Groq for data augmentation
subprocess.run(["pip", "install", "-q", "groq"], check=True)
 
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"
 
from unsloth import FastLanguageModel
import torch
 
print("=" * 60)
print("BioGPT: LLM-Based BioScript Compiler Automation")
print("=" * 60)
print(f"✅ Unsloth ready")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ GPU count: {torch.cuda.device_count()}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
 
for i in range(torch.cuda.device_count()):
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"   GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 24.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 kB 917.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/11

In [2]:
# ============================================================
# CELL 2 — CODE: Core Imports & Config
# ============================================================

import os, json, time, random
from pathlib import Path
from datetime import datetime
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
 
# Load secrets
HF_TOKEN   = UserSecretsClient().get_secret("HF_TOKEN")
GROQ_KEY   = UserSecretsClient().get_secret("GROQ_API_KEY")
 
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
 
# Paths
BASE_DIR   = Path("/kaggle/working")
DATA_DIR   = BASE_DIR / "biogpt_dataset"
AUG_DIR    = BASE_DIR / "biogpt_augmented"
MODELS_DIR = BASE_DIR / "biogpt_models"
 
for d in [DATA_DIR, AUG_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
 
# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
 
# Training config (shared across all models)
MAX_SEQ_LENGTH = 2048
EPOCHS         = 5
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
LR             = 2e-4
LORA_RANK      = 16
LORA_ALPHA     = 32
 
print("✅ Configuration set")
print(f"   Random seed    : {RANDOM_SEED}")
print(f"   Max seq length : {MAX_SEQ_LENGTH}")
print(f"   Epochs         : {EPOCHS}")
print(f"   Batch size     : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} effective")
print(f"   Learning rate  : {LR}")
print(f"   LoRA rank      : {LORA_RANK}")

✅ Configuration set
   Random seed    : 42
   Max seq length : 2048
   Epochs         : 5
   Batch size     : 2 × 4 = 8 effective
   Learning rate  : 0.0002
   LoRA rank      : 16


## 2. Dataset Loading & Exploration
 
**OpenBioSet** consists of 175 biological protocol folders, each containing:
- `description.txt` — Natural language protocol description (input)
- `*.bs` — BioScript source code (target output)
- `*.json` — Protocol metadata
- `output.ir` — Compiled intermediate representation
- `output.dot` — Control flow graph
 
We extract 174 valid (description, BioScript) pairs — one folder 
(`DMF_AP_FDP_Enzyme_Assay`) lacks a description file and is excluded.

In [3]:
# ============================================================
# CELL 3 — CODE: Data Extraction
# ============================================================

DATASET_PATH = "/kaggle/input/datasets/ishangain/openbioset/Dataset"
 
def find_file_by_suffix(folder: Path, extension: str):
    matches = [f for f in folder.iterdir()
               if f.is_file() and f.name.endswith(extension)]
    if not matches:
        return None
    for f in matches:
        if folder.name in f.name:
            return f
    return matches[0]
 
def find_description_file(folder: Path):
    candidates = [f for f in folder.iterdir()
                  if f.is_file() and "description" in f.name.lower()
                  and f.name.lower().endswith(".txt")]
    return candidates[0] if candidates else None
 
def extract_record(folder: Path) -> dict:
    record = {
        "folder_name"  : folder.name,
        "status"       : "ok",
        "errors"       : [],
        "description"  : None,
        "bioscript"    : None,
        "json_metadata": None,
    }
 
    desc_path = find_description_file(folder)
    if desc_path:
        text = desc_path.read_text(encoding="utf-8", errors="replace").strip()
        if text:
            record["description"] = text
        else:
            record["errors"].append("description.txt is empty")
    else:
        record["errors"].append("No description file found")
 
    bs_path = find_file_by_suffix(folder, ".bs")
    if bs_path:
        code = bs_path.read_text(encoding="utf-8", errors="replace").strip()
        if code:
            record["bioscript"] = code
        else:
            record["errors"].append(f"{bs_path.name} is empty")
    else:
        record["errors"].append("No .bs file found")
 
    json_path = find_file_by_suffix(folder, ".json")
    if json_path:
        try:
            record["json_metadata"] = json.loads(
                json_path.read_text(encoding="utf-8", errors="replace")
            )
        except:
            pass
 
    if record["errors"]:
        record["status"] = "error"
 
    return record
 
# Extract all records
dataset_root = Path(DATASET_PATH)
all_folders  = sorted([f for f in dataset_root.iterdir()
                        if f.is_dir() and not f.name.startswith(".")])
 
all_records   = [extract_record(f) for f in all_folders]
good_records  = [r for r in all_records if r["status"] == "ok"]
error_records = [r for r in all_records if r["status"] == "error"]
 
print(f"✅ Dataset Statistics:")
print(f"   Total folders    : {len(all_folders)}")
print(f"   Valid pairs      : {len(good_records)}")
print(f"   Excluded folders : {len(error_records)}")
if error_records:
    for r in error_records:
        print(f"     - {r['folder_name']}: {r['errors'][0]}")
 
desc_lens = [len(r["description"]) for r in good_records]
bs_lens   = [len(r["bioscript"])   for r in good_records]
 
print(f"\\n   Description length (chars):")
print(f"     Min / Max / Mean : {min(desc_lens):,} / {max(desc_lens):,} / {sum(desc_lens)//len(desc_lens):,}")
print(f"   BioScript length (chars):")
print(f"     Min / Max / Mean : {min(bs_lens):,} / {max(bs_lens):,} / {sum(bs_lens)//len(bs_lens):,}")

✅ Dataset Statistics:
   Total folders    : 175
   Valid pairs      : 174
   Excluded folders : 1
     - DMF_AP_FDP_Enzyme_Assay: No description file found
\n   Description length (chars):
     Min / Max / Mean : 756 / 22,242 / 9,941
   BioScript length (chars):
     Min / Max / Mean : 351 / 42,491 / 11,608


In [4]:
# ============================================================
# CELL 4 — CODE: Train/Val/Test Split
# ============================================================

def split_dataset(data, train=0.80, val=0.10):
    d = data[:]
    random.shuffle(d)
    n       = len(d)
    n_train = int(n * train)
    n_val   = int(n * val)
    return d[:n_train], d[n_train:n_train+n_val], d[n_train+n_val:]
 
train_records, val_records, test_records = split_dataset(good_records)
 
print(f"✅ Dataset Split (80/10/10):")
print(f"   Train : {len(train_records)} protocols")
print(f"   Val   : {len(val_records)} protocols")
print(f"   Test  : {len(test_records)} protocols")
 
# Save raw extraction
with open(DATA_DIR / "raw_extraction.json", "w") as f:
    json.dump(all_records, f, ensure_ascii=False, indent=2)
 
# Build Format C (instruction/chat format)
SYSTEM_PROMPT_BASE = (
    "You are BioGPT, an expert biological protocol compiler. "
    "Given a natural language description of a biological laboratory protocol, "
    "generate syntactically correct BioScript (.bs) code.\\n\\n"
    "You MUST always generate ALL sections in this exact order:\\n"
    "1. module declarations\\n"
    "2. manifest declarations\\n"
    "3. instructions: keyword\\n"
    "4. operations (dispense, mix, heat, detect, dispose)\\n\\n"
    "STRICT FORMAT EXAMPLE:\\n"
    "module myModule\\n\\n"
    "manifest Reagent1\\n"
    "manifest Reagent2\\n\\n"
    "instructions:\\n\\n"
    "// Step 1: description\\n"
    "var1 = dispense Reagent1 into $1 for 5s\\n"
    "var2 = dispense Reagent2 into $2 for 5s\\n"
    "mix1 = mix var1 with var2 for 30s\\n"
    "result = detect fluorescence on mix1 for 10s\\n\\n"
    "RULES:\\n"
    "- Always include instructions: section with actual operations\\n"
    "- Duration format: 5s or 5m (never 5h)\\n"
    "- Temperature format: 37c, 95c, 4c\\n"
    "- Output ONLY valid BioScript code. No explanations."
)
 
def make_example(record):
    return {
        "id": record["folder_name"],
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT_BASE},
            {"role": "user",      "content": record["description"]},
            {"role": "assistant", "content": record["bioscript"]},
        ]
    }
 
train_examples = [make_example(r) for r in train_records]
val_examples   = [make_example(r) for r in val_records]
test_examples  = [make_example(r) for r in test_records]
 
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\\n")
    print(f"   Saved {path.name}: {len(data)} examples")
 
print("\\n✅ Saving dataset splits:")
save_jsonl(train_examples, DATA_DIR / "train.jsonl")
save_jsonl(val_examples,   DATA_DIR / "val.jsonl")
save_jsonl(test_examples,  DATA_DIR / "test.jsonl")

✅ Dataset Split (80/10/10):
   Train : 139 protocols
   Val   : 17 protocols
   Test  : 18 protocols
\n✅ Saving dataset splits:
   Saved train.jsonl: 139 examples
   Saved val.jsonl: 17 examples
   Saved test.jsonl: 18 examples


## 3. Data Augmentation
 
To address the limited dataset size (139 training examples), we apply 
**LLM-based data augmentation** using the Groq API (Llama-3.3-70B-Versatile).
 
For each training protocol description, we generate **3 paraphrased variants** 
with different writing styles:
- **Variant 1:** Concise and technical
- **Variant 2:** Step-by-step focused
- **Variant 3:** Context and rationale focused
 
The BioScript code remains **identical** for all variants — only the natural 
language input is diversified. This teaches the model to handle different 
writing styles for the same protocol.
 
**Result:** 139 originals × 4 versions = **556 training examples** (4x increase)

In [6]:
# ============================================================
# CELL 5 — CODE: Data Augmentation
# ============================================================

from groq import Groq
import json
import time
import random
 
groq_client    = Groq(api_key=GROQ_KEY)
GROQ_MODEL     = "llama-3.3-70b-versatile"
SLEEP_BETWEEN  = 3
PROGRESS_FILE  = AUG_DIR / "progress.json"
 
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]
 
def paraphrase_all_variants(description: str, folder_name: str) -> list:
    desc   = description[:2500] + "..." if len(description) > 2500 else description
    
    # FIXED: Using triple quotes for the multiline f-string
    prompt = f"""Rewrite this biology lab protocol description in 3 styles.
Keep all scientific details identical.
 
Return ONLY this exact JSON format:
{{"v1": "concise technical version", "v2": "step by step version", "v3": "context focused version"}}
 
DESCRIPTION:
{desc}"""
 
    for attempt in range(3):
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2500,
                temperature=0.5
            )
            text = response.choices[0].message.content.strip()
            text = text.replace("```json", "").replace("```", "").strip()
            start, end = text.find("{"), text.rfind("}") + 1
            if start >= 0 and end > start:
                text = text[start:end]
            data     = json.loads(text)
            variants = [data.get("v1",""), data.get("v2",""), data.get("v3","")]
            if all(len(v) > 50 for v in variants):
                return variants
        except json.JSONDecodeError:
            time.sleep(5)
        except Exception as e:
            time.sleep((2 ** attempt) * 10)
 
    return [description, description, description]
 
# FIXED: Create the directory if it doesn't exist after the reset
AUG_DIR.mkdir(parents=True, exist_ok=True)

# Load progress (resume if interrupted)
progress = json.loads(PROGRESS_FILE.read_text()) if PROGRESS_FILE.exists() else {}
 
print(f"Starting augmentation...")
print(f"   Training records  : {len(train_examples)}")
print(f"   Variants per rec  : 3")
print(f"   Already done      : {len([k for k in progress if k.endswith('__v0')])}")
 
augmented_variants = []
completed = skipped = 0
 
for record in train_records:
    folder_name = record["folder_name"]
    prog_key    = f"{folder_name}__v0"
 
    if prog_key in progress:
        for v_idx in range(3):
            k = f"{folder_name}__v{v_idx}"
            if k in progress:
                augmented_variants.append((folder_name, v_idx, progress[k]))
        skipped += 1
        continue
 
    # NEW: Added visual feedback so you know it isn't hanging during the API call
    print(f"   Calling API for record: {folder_name}...")
    
    variants = paraphrase_all_variants(record["description"], folder_name)
 
    for v_idx, variant_text in enumerate(variants):
        k = f"{folder_name}__v{v_idx}"
        progress[k] = variant_text
        augmented_variants.append((folder_name, v_idx, variant_text))
 
    PROGRESS_FILE.write_text(json.dumps(progress, indent=2))
    completed += 1
 
    if completed % 10 == 0:
        print(f"   Progress: {completed + skipped}/{len(train_records)} records")
 
    time.sleep(SLEEP_BETWEEN)
 
print(f"\n✅ Augmentation complete!")
print(f"   New variants    : {completed * 3}")
print(f"   Cached variants : {skipped * 3}")
print(f"   Total variants  : {len(augmented_variants)}")
 
# Build augmented training set
bs_lookup = {r["folder_name"]: r["bioscript"] for r in good_records}
aug_train = []
 
for record in train_records:
    aug_train.append({
        "id": record["folder_name"] + "__original",
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT_BASE},
            {"role": "user",      "content": record["description"]},
            {"role": "assistant", "content": record["bioscript"]},
        ],
        "source": "original"
    })
 
for folder_name, v_idx, paraphrased in augmented_variants:
    bs = bs_lookup.get(folder_name, "")
    if not bs:
        continue
    aug_train.append({
        "id": f"{folder_name}__v{v_idx+1}",
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT_BASE},
            {"role": "user",      "content": paraphrased},
            {"role": "assistant", "content": bs},
        ],
        "source": f"augmented_v{v_idx+1}"
    })
 
random.shuffle(aug_train)
 
print(f"\n✅ Final training set: {len(aug_train)} examples")
print(f"   Originals  : {len(train_records)}")
print(f"   Augmented  : {len(augmented_variants)}")
 
save_jsonl(aug_train,    AUG_DIR / "train_augmented.jsonl")
save_jsonl(val_examples, AUG_DIR / "val.jsonl")
save_jsonl(test_examples,AUG_DIR / "test.jsonl")

Starting augmentation...
   Training records  : 139
   Variants per rec  : 3
   Already done      : 0
   Calling API for record: xia_proxseq_computational_prediction...
   Calling API for record: patel_nfkb_computer_vision_single_cell...
   Calling API for record: schwarz_chemokine_migration_microfluidics...
   Calling API for record: mei_dmf_plasma_protein_depletion...
   Calling API for record: Lengen_Lentiviral...
   Calling API for record: Oxycodone...
   Calling API for record: kim2013_ngs_library_prep_dmf...
   Calling API for record: Mammalian_Cell_Culture_Platform...
   Calling API for record: warkiani_spiral_ctc_enrichment...
   Calling API for record: kim_immunomodulator_hts...
   Progress: 10/139 records
   Calling API for record: ahmadi_2024_mab_discovery...
   Calling API for record: Mage_Electrotransformation...
   Calling API for record: son_nfkb_dose_differentiation...
   Calling API for record: Diazepam...
   Calling API for record: Aerosol_Sampling...
   Calling API f

### Dataset Reconstruction from Progress Cache

If JSONL files are corrupted, we reconstruct the training set directly
from the saved augmentation progress file (`progress.json`).
This ensures no augmentation work is lost even if file writing fails.

In [17]:
# ============================================================
# CELL 6 — CODE: Dataset Recovery
# ============================================================

# Reconstructs train_augmented.jsonl, val.jsonl, and test.jsonl
# from the saved progress.json file using the same random seed (42)
# to ensure identical train/val/test splits.

import json, random
from pathlib import Path

DATA_DIR  = Path("/kaggle/working/biogpt_dataset")
AUG_DIR   = Path("/kaggle/working/biogpt_augmented")
PROGRESS_FILE = AUG_DIR / "progress.json"

# Load raw extraction
with open(DATA_DIR / "raw_extraction.json", "r") as f:
    all_records = json.load(f)

good_records = [r for r in all_records if r["status"] == "ok"]

# Load progress (augmented variants)
with open(PROGRESS_FILE, "r") as f:
    progress = json.load(f)

print(f"✅ Loaded {len(good_records)} records")
print(f"✅ Loaded {len(progress)} augmented variants from progress file")

# Rebuild splits using same seed
random.seed(42)
shuffled = good_records[:]
random.shuffle(shuffled)
n = len(shuffled)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)
train_records = shuffled[:n_train]
val_records   = shuffled[n_train:n_train+n_val]
test_records  = shuffled[n_train+n_val:]

print(f"\nSplit: Train={len(train_records)} Val={len(val_records)} Test={len(test_records)}")

SYSTEM_PROMPT = (
    "You are BioGPT, an expert biological protocol compiler. "
    "Given a natural language description of a biological laboratory protocol, "
    "generate syntactically correct BioScript (.bs) code.\n\n"
    "You MUST always generate ALL sections in this exact order:\n"
    "1. module declarations\n"
    "2. manifest declarations\n"
    "3. instructions: keyword\n"
    "4. operations (dispense, mix, heat, detect, dispose)\n\n"
    "STRICT FORMAT EXAMPLE:\n"
    "module myModule\n\n"
    "manifest Reagent1\n"
    "manifest Reagent2\n\n"
    "instructions:\n\n"
    "// Step 1: description\n"
    "var1 = dispense Reagent1 into $1 for 5s\n"
    "var2 = dispense Reagent2 into $2 for 5s\n"
    "mix1 = mix var1 with var2 for 30s\n"
    "result = detect fluorescence on mix1 for 10s\n\n"
    "RULES:\n"
    "- Always include instructions: section with actual operations\n"
    "- Duration format: 5s or 5m (never 5h)\n"
    "- Temperature format: 37c, 95c, 4c\n"
    "- Output ONLY valid BioScript code. No explanations."
)

def make_example(record, source="original"):
    return {
        "id": record["folder_name"] + f"__{source}",
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": record["description"]},
            {"role": "assistant", "content": record["bioscript"]},
        ],
        "source": source
    }

# Build augmented training set
bs_lookup = {r["folder_name"]: r["bioscript"] for r in good_records}
aug_train = []

# Add originals
for record in train_records:
    aug_train.append(make_example(record, "original"))

# Add augmented variants from progress
for folder_name in [r["folder_name"] for r in train_records]:
    for v_idx in range(3):
        k = f"{folder_name}__v{v_idx}"
        if k in progress:
            paraphrased = progress[k]
            bs = bs_lookup.get(folder_name, "")
            if bs and paraphrased != record.get("description", ""):
                aug_train.append({
                    "id": f"{folder_name}__v{v_idx+1}",
                    "messages": [
                        {"role": "system",    "content": SYSTEM_PROMPT},
                        {"role": "user",      "content": paraphrased},
                        {"role": "assistant", "content": bs},
                    ],
                    "source": f"augmented_v{v_idx+1}"
                })

random.shuffle(aug_train)

val_examples  = [make_example(r) for r in val_records]
test_examples = [make_example(r) for r in test_records]

# Save properly as JSONL
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    # Verify
    with open(path, "r") as f:
        count = sum(1 for line in f if line.strip())
    print(f"  ✅ {path.name}: {count} lines saved")
    return count

print("\nSaving JSONL files:")
save_jsonl(aug_train,    AUG_DIR / "train_augmented.jsonl")
save_jsonl(val_examples, AUG_DIR / "val.jsonl")
save_jsonl(test_examples,AUG_DIR / "test.jsonl")

print(f"\n✅ Done! Training set: {len(aug_train)} examples")

✅ Loaded 174 records
✅ Loaded 417 augmented variants from progress file

Split: Train=139 Val=17 Test=18

Saving JSONL files:
  ✅ train_augmented.jsonl: 553 lines saved
  ✅ val.jsonl: 17 lines saved
  ✅ test.jsonl: 18 lines saved

✅ Done! Training set: 553 examples


## 4. Model Training
 
We fine-tune 4 models using **QLoRA** (Quantized Low-Rank Adaptation):
- 4-bit quantization reduces memory requirements
- LoRA adapters train only ~0.5-0.75% of parameters
- Unsloth provides 2x speed improvement over standard QLoRA
 
**Models compared:**
 
| Model | Parameters | Epochs | Architecture |
|-------|-----------|--------|--------------|
| Qwen2.5-Coder-3B-Instruct | 3B | 3 | Code-specialized |
| Qwen2.5-Coder-3B-Instruct | 3B | 5 | Code-specialized |
| Llama-3.2-3B-Instruct | 3B | 5 | General instruction |
| Qwen2.5-Coder-7B-Instruct | 7B | 5 | Code-specialized |
 
**LoRA Configuration (shared across all models):**
- Rank (r): 16
- Alpha: 32
- Target modules: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
- Dropout: 0.05

In [6]:
# ============================================================
# CELL 7 — CODE: Training Function
# ============================================================

from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import json
import torch
from datetime import datetime
 
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]
 
def train_model(model_id: str, output_name: str,
                epochs: int, batch_size: int = 2, grad_accum: int = 4):
    """
    Fine-tune a model using QLoRA and save the LoRA adapter.
    Returns training loss history for reporting.
    """
    print(f"\n{'='*60}")
    print(f"Training: {model_id}")
    print(f"Epochs  : {epochs} | Output: {output_name}")
    print(f"{'='*60}")
 
    output_dir = MODELS_DIR / output_name
    output_dir.mkdir(parents=True, exist_ok=True)
 
    # Load model
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        token=HF_TOKEN,
    )
 
    # Verify tokenization
    test_text = "module pcr\n\nmanifest DNA\n\ninstructions:\n\nbuffer = dispense DNA into $1 for 5s"
    decoded   = tokenizer.decode(
        tokenizer(test_text, return_tensors="pt")["input_ids"][0],
        skip_special_tokens=True
    )
    assert " " in decoded, f"Tokenizer strips spaces for {model_id}!"
    print(f"✅ Tokenization verified")
 
    # Apply LoRA
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=RANDOM_SEED,
    )
    model.print_trainable_parameters()
 
    # Format dataset
    train_data = load_jsonl(AUG_DIR / "train_augmented.jsonl")
    val_data   = load_jsonl(AUG_DIR / "val.jsonl")
 
    def format_example(example):
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
        return {"text": text}
 
    train_dataset = Dataset.from_list([format_example(ex) for ex in train_data])
    val_dataset   = Dataset.from_list([format_example(ex) for ex in val_data])
 
    # Train
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        args=SFTConfig(
            output_dir=str(output_dir / "checkpoints"),
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LENGTH,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            gradient_accumulation_steps=grad_accum,
            learning_rate=LR,
            warmup_ratio=0.05,
            lr_scheduler_type="cosine",
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=50,
            save_strategy="steps",
            save_steps=50,
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            optim="adamw_8bit",
            report_to="none",
            packing=False,
            dataset_num_proc=2,
            seed=RANDOM_SEED,
        ),
    )
 
    started = datetime.now()
    print(f"\n🚀 Started: {started.strftime('%Y-%m-%d %H:%M:%S')}")
    trainer.train()
    finished = datetime.now()
    duration = (finished - started).seconds // 60
 
    print(f"\n✅ Finished: {finished.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Duration: {duration} minutes")
 
    # Save adapter
    adapter_path = output_dir / "lora_adapter"
    model.save_pretrained(str(adapter_path))
    tokenizer.save_pretrained(str(adapter_path))
    size = sum(f.stat().st_size for f in adapter_path.rglob("*")) / 1e6
    print(f"✅ Adapter saved: {size:.1f} MB → {adapter_path}")
 
    # Save summary
    summary = {
        "model_id"    : model_id,
        "output_name" : output_name,
        "epochs"      : epochs,
        "duration_min": duration,
        "lora_rank"   : LORA_RANK,
        "lora_alpha"  : LORA_ALPHA,
        "train_size"  : len(train_data),
        "val_size"    : len(val_data),
        "finished_at" : finished.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open(output_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
 
    # Free GPU memory
    del model, tokenizer
    torch.cuda.empty_cache()
    print(f"✅ GPU memory freed")
 
    return summary
 
print("✅ Training function ready.")

✅ Training function ready.


In [11]:
# ============================================================
# CELL 8 — Training Model 1: Qwen2.5-Coder-3B (3 epochs)
# ============================================================

# ── Model 1: Qwen2.5-Coder-3B-Instruct (3 epochs) — Baseline ────────────────
training_summaries = {}

training_summaries["Qwen2.5-Coder-3B-3ep"] = train_model(
    model_id    = "Qwen/Qwen2.5-Coder-3B-Instruct",
    output_name = "qwen3b_3ep",
    epochs      = 3,
    batch_size  = 2,
    grad_accum  = 4,
)

print(f"\n✅ Model 1 complete!")
print(f"   Duration: {training_summaries['Qwen2.5-Coder-3B-3ep']['duration_min']} minutes")


Training: Qwen/Qwen2.5-Coder-3B-Instruct
Epochs  : 3 | Output: qwen3b_3ep
==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Tokenization verified


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


...

Step,Training Loss,Validation Loss
50,1.234386,1.326451
100,0.910566,1.465452
150,0.608899,1.653340
200,0.530449,1.753500
210,0.480982,1.750433


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen3b_3ep/checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen3b_3ep/checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen3b_3ep/checkpoints/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen3b_3ep/checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen3b_3ep/checkpoints/checkpoint-210/tokenizer_config.json.



✅ Finished: 2026-08-20 08:22:25
   Duration: 84 minutes


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen3b_3ep/lora_adapter/tokenizer_config.json.


✅ Adapter saved: 131.2 MB → /kaggle/working/biogpt_models/qwen3b_3ep/lora_adapter
✅ GPU memory freed

✅ Model 1 complete!
   Duration: 84 minutes


In [ ]:
# ============================================================
# CELL 9 — Training Model 2: Qwen2.5-Coder-3B (5 epochs)
# ============================================================

# ── Model 2: Qwen2.5-Coder-3B-Instruct (5 epochs) ───────────────────────────
training_summaries["Qwen2.5-Coder-3B-5ep"] = train_model(
    model_id    = "Qwen/Qwen2.5-Coder-3B-Instruct",
    output_name = "qwen3b_5ep",
    epochs      = 5,
    batch_size  = 2,
    grad_accum  = 4,
)

print(f"\n✅ Model 2 complete!")
print(f"   Duration: {training_summaries['Qwen2.5-Coder-3B-5ep']['duration_min']} minutes")


Training: Qwen/Qwen2.5-Coder-3B-Instruct
Epochs  : 5 | Output: qwen3b_5ep
==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


✅ Tokenization verified


Unsloth 2026.8.18 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.

🚀 Started: 2026-08-20 08:49:26


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 553 | Num Epochs = 5 | Total steps = 350
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,1.252365,1.329619
100,0.908527,1.466160
150,0.498821,1.746168
200,0.291469,2.133053
250,0.106548,2.463501
300,0.060008,2.734588


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen3b_5ep/checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen3b_5ep/checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen3b_5ep/checkpoints/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen3b_5ep/checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen3b_5ep/checkpoints/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen3b_5ep/checkpoints/checkpoint-300/tokenizer_config.json.


In [5]:
# ============================================================
# CELL 10 — Training Model 3: Llama-3.2-3B (5 epochs)
# ============================================================

# ── Model 3: Llama-3.2-3B-Instruct (5 epochs) ───────────────────────────────
training_summaries["Llama-3.2-3B-5ep"] = train_model(
    model_id    = "meta-llama/Llama-3.2-3B-Instruct",
    output_name = "llama3b_5ep",
    epochs      = 5,
    batch_size  = 2,
    grad_accum  = 4,
)

print(f"\n✅ Model 3 complete!")
print(f"   Duration: {training_summaries['Llama-3.2-3B-5ep']['duration_min']} minutes")


Training: meta-llama/Llama-3.2-3B-Instruct
Epochs  : 5 | Output: llama3b_5ep
==((====))==  Unsloth 2026.8.19: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


✅ Tokenization verified


Unsloth 2026.8.19 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/llama3b_5ep/checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/llama3b_5ep/checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/llama3b_5ep/checkpoints/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/llama3b_5ep/checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/llama3b_5ep/checkpoints/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/llama3b_5ep/checkpoints/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/llama3b_5ep/checkpoints/checkpoint-350/tokenizer_config.json.



✅ Finished: 2026-08-21 08:32:15
   Duration: 122 minutes


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/llama3b_5ep/lora_adapter/tokenizer_config.json.


✅ Adapter saved: 114.6 MB → /kaggle/working/biogpt_models/llama3b_5ep/lora_adapter
✅ GPU memory freed


NameError: name 'training_summaries' is not defined

In [7]:
# ============================================================
# CELL 11 — Training Model 4: Qwen2.5-Coder-7B (5 epochs)
# ============================================================

# ── Model 4: Qwen2.5-Coder-7B-Instruct (5 epochs) — Best expected ────────────
training_summaries["Qwen2.5-Coder-7B-5ep"] = train_model(
    model_id    = "Qwen/Qwen2.5-Coder-7B-Instruct",
    output_name = "qwen7b_5ep",
    epochs      = 5,
    batch_size  = 1,
    grad_accum  = 8,
)

print(f"\n✅ Model 4 complete!")
print(f"   Duration: {training_summaries['Qwen2.5-Coder-7B-5ep']['duration_min']} minutes")


Training: Qwen/Qwen2.5-Coder-7B-Instruct
Epochs  : 5 | Output: qwen7b_5ep
==((====))==  Unsloth 2026.8.19: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


✅ Tokenization verified


Unsloth 2026.8.19 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen7b_5ep/checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen7b_5ep/checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen7b_5ep/checkpoints/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen7b_5ep/checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen7b_5ep/checkpoints/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen7b_5ep/checkpoints/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen7b_5ep/checkpoints/checkpoint-350/tokenizer_config.json.



✅ Finished: 2026-08-22 11:54:53
   Duration: 286 minutes


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/biogpt_models/qwen7b_5ep/lora_adapter/tokenizer_config.json.


✅ Adapter saved: 173.0 MB → /kaggle/working/biogpt_models/qwen7b_5ep/lora_adapter
✅ GPU memory freed


NameError: name 'training_summaries' is not defined

In [10]:
# ============================================================
# CELL 12 - CODE: Models Check (Saved or Not)
# ============================================================

from pathlib import Path

models = {
    "Qwen-3B (3ep)" : "/kaggle/working/biogpt_models/qwen3b_3ep/lora_adapter",
    "Qwen-3B (5ep)" : "/kaggle/working/biogpt_models/qwen3b_5ep/lora_adapter",
    "Llama-3B (5ep)" : "/kaggle/working/biogpt_models/llama3b_5ep/lora_adapter",
    "Qwen-7B (5ep)" : "/kaggle/working/biogpt_models/qwen7b_5ep/lora_adapter",
}

for name, path in models.items():
    p = Path(path)
    if p.exists():
        size = sum(f.stat().st_size for f in p.rglob("*")) / 1e6
        print(f"✅ {name}: {size:.1f} MB")
    else:
        print(f"❌ {name}: NOT FOUND — need to retrain")

✅ Qwen-3B (3ep): 131.2 MB
✅ Qwen-3B (5ep): 131.2 MB
✅ Llama-3B (5ep): 114.6 MB
✅ Qwen-7B (5ep): 173.0 MB


## 5. Evaluation
 
We evaluate all 4 models on the **18 held-out test protocols** using:
 
1. **Structural Accuracy** — Does the generated code contain required BioScript elements?
   - `module` declarations
   - `manifest` declarations  
   - `instructions:` section
   - Operations (`dispense`, `mix`, etc.)
   - No repetitive output
   - Non-empty output
 
2. **BLEU-1 Score** — Word-level overlap between generated and reference BioScript
 
**Generation parameters:**
- Temperature: 0.7
- Top-p: 0.9, Top-k: 50
- Repetition penalty: 1.3
- Max new tokens: 1024

In [2]:
# ============================================================
# CELL 13 — CODE: setup cell
# ============================================================

import os, json, torch
from pathlib import Path
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

DATA_DIR   = Path("/kaggle/working/biogpt_augmented")
MODELS_DIR = Path("/kaggle/working/biogpt_models")
EVAL_DIR   = Path("/kaggle/working/evaluation")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LEN = 2048

EVAL_MODELS = {
    "Qwen2.5-Coder-3B-3ep": str(MODELS_DIR / "qwen3b_3ep/lora_adapter"),
    "Qwen2.5-Coder-3B-5ep": str(MODELS_DIR / "qwen3b_5ep/lora_adapter"),
    "Llama-3.2-3B-5ep"    : str(MODELS_DIR / "llama3b_5ep/lora_adapter"),
    "Qwen2.5-Coder-7B-5ep": str(MODELS_DIR / "qwen7b_5ep/lora_adapter"),
}

print(f"✅ Config ready")
print(f"✅ MAX_SEQ_LEN: {MAX_SEQ_LEN}")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

✅ Config ready
✅ MAX_SEQ_LEN: 2048
✅ GPU: Tesla T4


In [3]:
# ============================================================
# CELL 14 — CODE: Evaluation Functions
# ============================================================

SYSTEM_PROMPT = (
    "You are BioGPT, an expert biological protocol compiler. "
    "Given a natural language description of a biological laboratory protocol, "
    "generate syntactically correct BioScript (.bs) code.\n\n"
    "You MUST always generate ALL sections in this exact order:\n"
    "1. module declarations\n"
    "2. manifest declarations\n"
    "3. instructions: keyword\n"
    "4. operations (dispense, mix, heat, detect, dispose)\n\n"
    "STRICT FORMAT EXAMPLE:\n"
    "module myModule\n\n"
    "manifest Reagent1\n"
    "manifest Reagent2\n\n"
    "instructions:\n\n"
    "// Step 1: description\n"
    "var1 = dispense Reagent1 into $1 for 5s\n"
    "var2 = dispense Reagent2 into $2 for 5s\n"
    "mix1 = mix var1 with var2 for 30s\n"
    "result = detect fluorescence on mix1 for 10s\n\n"
    "RULES:\n"
    "- Always include instructions: section with actual operations\n"
    "- Duration format: 5s or 5m (never 5h)\n"
    "- Temperature format: 37c, 95c, 4c\n"
    "- Output ONLY valid BioScript code. No explanations."
)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def _is_repetitive(text):
    words = text.split()
    if len(words) < 10:
        return False
    for i in range(len(words) - 3):
        phrase = " ".join(words[i:i+3])
        if text.count(phrase) > len(words) * 0.1:
            return True
    return False

def check_structure(code):
    full  = code.lower()
    lines = full.split("\n")
    checks = {
        "has_module"      : any("module" in l for l in lines),
        "has_manifest"    : any("manifest" in l for l in lines),
        "has_instructions": "instructions:" in full,
        "has_dispense"    : "dispense" in full,
        "has_mix"         : "mix" in full,
        "has_braces"      : "{" in code and "}" in code,
        "no_repetition"   : not _is_repetitive(code),
        "non_empty"       : len(code.strip()) > 50,
    }
    num_checks      = len(checks)
    checks["score"] = sum(checks.values()) / num_checks
    return checks

def compute_bleu(reference, hypothesis):
    ref_words = set(reference.lower().split())
    hyp_words = hypothesis.lower().split()
    if not hyp_words:
        return 0.0
    return sum(1 for w in hyp_words if w in ref_words) / len(hyp_words)

def generate_bioscript(model, tokenizer, description, max_new_tokens=1024):
    desc   = description[:1500] + "..." if len(description) > 1500 else description
    chat   = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": desc},
    ]
    prompt = tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN - max_new_tokens,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()

print("✅ Evaluation functions ready.")

✅ Evaluation functions ready.


In [25]:
# ============================================================
# CELL 15 - CODE: Validation Check 
# ============================================================

from pathlib import Path

# Check which path the evaluation cell is using
DATA_DIR = Path("/kaggle/working/biogpt_augmented")

# Verify the file is correct
with open(DATA_DIR / "test.jsonl", "r") as f:
    lines = f.readlines()

print(f"Total lines in test.jsonl: {len(lines)}")
print(f"First line length: {len(lines[0])} chars")

# Try parsing each line
for i, line in enumerate(lines):
    try:
        import json
        json.loads(line.strip())
    except Exception as e:
        print(f"❌ Line {i+1} is bad: {e}")

print("✅ All lines valid!" if all(True for line in lines) else "")

Total lines in test.jsonl: 18
First line length: 41438 chars
✅ All lines valid!


In [4]:
# ============================================================
# CELL 16 — CODE: reset the path and reload
# ============================================================

import json
from pathlib import Path

# Force correct path
DATA_DIR = Path("/kaggle/working/biogpt_augmented")

def load_jsonl(path):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

# Verify
test_data = load_jsonl(DATA_DIR / "test.jsonl")
print(f"✅ test.jsonl loaded: {len(test_data)} examples")

train_data = load_jsonl(DATA_DIR / "train_augmented.jsonl")
print(f"✅ train_augmented.jsonl loaded: {len(train_data)} examples")

val_data = load_jsonl(DATA_DIR / "val.jsonl")
print(f"✅ val.jsonl loaded: {len(val_data)} examples")

✅ test.jsonl loaded: 18 examples
✅ train_augmented.jsonl loaded: 553 examples
✅ val.jsonl loaded: 17 examples


In [5]:
# ============================================================
# CELL 17 — CODE: Run Evaluation
# ============================================================

test_data   = load_jsonl(DATA_DIR / "test.jsonl")
all_results = {}

for model_name, adapter_path in EVAL_MODELS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=adapter_path,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
        token=os.environ["HF_TOKEN"],
    )
    FastLanguageModel.for_inference(model)

    results = []
    print(f"\n{'#':<4} {'ID':<42} {'Struct%':<10} {'BLEU':<8} {'Status'}")
    print("-" * 72)

    for i, example in enumerate(test_data):
        folder_name  = example["id"].replace("__original", "")
        description  = example["messages"][1]["content"]
        reference_bs = example["messages"][2]["content"]

        generated     = generate_bioscript(model, tokenizer, description)
        struct_checks = check_structure(generated)
        bleu_score    = compute_bleu(reference_bs, generated)
        is_rep        = not struct_checks["no_repetition"]
        status        = "⚠️" if is_rep else "✅"

        short_id = folder_name[:40] + ".." if len(folder_name) > 40 else folder_name
        print(f"{i+1:<4} {short_id:<42} "
              f"{struct_checks['score']*100:<10.1f} "
              f"{bleu_score:<8.3f} {status}")

        results.append({
            "id"           : folder_name,
            "generated"    : generated,
            "reference"    : reference_bs,
            "struct_checks": struct_checks,
            "bleu_score"   : bleu_score,
            "struct_score" : struct_checks["score"],
            "is_repetitive": is_rep,
        })

    all_results[model_name] = results

    with open(EVAL_DIR / f"{model_name}_results.json", "w") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    del model, tokenizer
    torch.cuda.empty_cache()
    print(f"\n✅ {model_name} done — GPU freed.")

print(f"\n✅ All models evaluated!")


Evaluating: Qwen2.5-Coder-3B-3ep
==((====))==  Unsloth 2026.8.19: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.8.19 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



#    ID                                         Struct%    BLEU     Status
------------------------------------------------------------------------


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

15   Picoliter_Droplet_Manipulation             100.0      0.146    ✅


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


16   Apoptosis                                  87.5       0.380    ✅


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


17   Estimation_of_Lipid_Peroxides              75.0       0.227    ✅
18   warkiani_membraneless_microfiltration      75.0       0.372    ✅

✅ Qwen2.5-Coder-7B-5ep done — GPU freed.

✅ All models evaluated!


## 6. Results & Comparison

The following table presents the evaluation results of all 4 fine-tuned 
BioGPT models on 18 held-out test protocols.

**Metrics explanation:**
- **Struct%** — Average structural accuracy (8 checks, higher is better)
- **BLEU** — Average BLEU-1 score (word overlap with reference)
- **Module/Manifest/Instruct/Dispense** — Coverage out of 18 test protocols
- **Non-Rep** — Non-repetitive outputs (18/18 = perfect for all models)

In [6]:
# ============================================================
# CELL 18 — CODE: Final Comparison Table
# ============================================================

# Summarizes all metrics across 4 models.
# Best model selected using weighted score:
# 40% structural accuracy + 30% BLEU + 30% instructions coverage

print("\n" + "=" * 95)
print("FINAL RESULTS: BioGPT Model Comparison on 18 Test Protocols")
print("=" * 95)
print(f"{'Model':<28} {'Struct%':>8} {'BLEU':>8} {'Module':>8} "
      f"{'Manifest':>10} {'Instruct':>10} {'Dispense':>10} {'Non-Rep':>9}")
print("-" * 95)

summary = {}
for model_name, results in all_results.items():
    total        = len(results)
    avg_struct   = sum(r["struct_score"] for r in results) / total * 100
    avg_bleu     = sum(r["bleu_score"] for r in results) / total
    has_module   = sum(1 for r in results if r["struct_checks"]["has_module"])
    has_manifest = sum(1 for r in results if r["struct_checks"]["has_manifest"])
    has_instruct = sum(1 for r in results if r["struct_checks"]["has_instructions"])
    has_dispense = sum(1 for r in results if r["struct_checks"]["has_dispense"])
    non_rep      = sum(1 for r in results if not r["is_repetitive"])

    summary[model_name] = {
        "avg_struct"  : avg_struct,
        "avg_bleu"    : avg_bleu,
        "has_module"  : has_module,
        "has_manifest": has_manifest,
        "has_instruct": has_instruct,
        "has_dispense": has_dispense,
        "non_rep"     : non_rep,
        "total"       : total,
    }

    print(f"{model_name:<28} {avg_struct:>8.1f} {avg_bleu:>8.3f} "
          f"{has_module}/{total:>5} {has_manifest}/{total:>7} "
          f"{has_instruct}/{total:>7} {has_dispense}/{total:>7} "
          f"{non_rep}/{total:>6}")

print("=" * 95)

best = max(summary, key=lambda x: (
    summary[x]["avg_struct"] * 0.4 +
    summary[x]["avg_bleu"] * 100 * 0.3 +
    (summary[x]["has_instruct"] / summary[x]["total"]) * 100 * 0.3
))

print(f"\n🏆 Best Model: {best}")
print(f"   Structural Accuracy : {summary[best]['avg_struct']:.1f}%")
print(f"   BLEU-1 Score        : {summary[best]['avg_bleu']:.3f}")
print(f"   Instructions        : {summary[best]['has_instruct']}/{summary[best]['total']}")

with open(EVAL_DIR / "comparison_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Results saved to: {EVAL_DIR}")


FINAL RESULTS: BioGPT Model Comparison on 18 Test Protocols
Model                         Struct%     BLEU   Module   Manifest   Instruct   Dispense   Non-Rep
-----------------------------------------------------------------------------------------------
Qwen2.5-Coder-3B-3ep             67.4    0.223 14/   18 10/     18 6/     18 10/     18 18/    18
Qwen2.5-Coder-3B-5ep             67.4    0.261 15/   18 13/     18 7/     18 9/     18 18/    18
Llama-3.2-3B-5ep                 73.6    0.344 14/   18 9/     18 9/     18 12/     18 18/    18
Qwen2.5-Coder-7B-5ep             88.2    0.354 17/   18 14/     18 12/     18 14/     18 18/    18

🏆 Best Model: Qwen2.5-Coder-7B-5ep
   Structural Accuracy : 88.2%
   BLEU-1 Score        : 0.354
   Instructions        : 12/18

✅ Results saved to: /kaggle/working/evaluation


## 7. Inference Demo

The following demonstrates the best-performing model 
(**Qwen2.5-Coder-7B-5ep**) generating BioScript code from 
natural language protocol descriptions on 3 unseen test protocols.

In [9]:
# ============================================================
# CELL 18 — CODE: Inference Demo — Best Model (Qwen2.5-Coder-7B)
# ============================================================

# Demonstrates end-to-end generation: natural language → BioScript code

demo_model, demo_tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(MODELS_DIR / "qwen7b_5ep/lora_adapter"),
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
    token=os.environ["HF_TOKEN"],
)
FastLanguageModel.for_inference(demo_model)

# Enhanced system prompt for inference
INFERENCE_PROMPT = (
    "You are BioGPT, an expert biological protocol compiler. "
    "Given a natural language description of a biological laboratory protocol, "
    "generate syntactically correct BioScript (.bs) code.\n\n"
    "You MUST always generate ALL sections in this exact order:\n"
    "1. module declarations\n"
    "2. manifest declarations\n"
    "3. instructions: keyword\n"
    "4. operations (dispense, mix, heat, detect, dispose)\n\n"
    "CRITICAL: Output ONLY valid BioScript code.\n"
    "DO NOT output JSON, XML, Python, or any other format.\n"
    "DO NOT output explanations or comments outside the code.\n\n"
    "CORRECT FORMAT:\n"
    "module moduleName\n\n"
    "manifest Reagent1\n"
    "manifest Reagent2\n\n"
    "instructions:\n\n"
    "// Step description\n"
    "var1 = dispense Reagent1 into $1 for 5s\n"
    "var2 = dispense Reagent2 into $2 for 5s\n"
    "mix1 = mix var1 with var2 for 30s\n"
    "result = detect fluorescence on mix1 for 10s\n\n"
    "WRONG FORMAT (never do this):\n"
    "```json\n"
    "{ 'operations': [...] }\n"
    "```\n"
)

def generate_bioscript_demo(description: str, max_new_tokens: int = 1024) -> str:
    desc   = description[:1500] + "..." if len(description) > 1500 else description
    chat   = [
        {"role": "system", "content": INFERENCE_PROMPT},
        {"role": "user",   "content": desc},
    ]
    prompt = demo_tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True,
    )
    # Seed with "module" to force BioScript format
    prompt += "module "

    inputs = demo_tokenizer(
        prompt, return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN - max_new_tokens,
    ).to("cuda")

    with torch.no_grad():
        outputs = demo_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.3,
            pad_token_id=demo_tokenizer.eos_token_id,
            eos_token_id=demo_tokenizer.eos_token_id,
        )

    generated = demo_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()

    # Prepend "module " since we seeded with it
    return "module " + generated

test_data = load_jsonl(DATA_DIR / "test.jsonl")

for i, example in enumerate(test_data[:3]):
    protocol_id  = example["id"].replace("__original", "")
    description  = example["messages"][1]["content"]
    reference_bs = example["messages"][2]["content"]

    generated = generate_bioscript_demo(description)
    checks    = check_structure(generated)
    bleu      = compute_bleu(reference_bs, generated)

    print(f"\n{'='*70}")
    print(f"PROTOCOL {i+1}: {protocol_id}")
    print(f"{'='*70}")
    print(f"\n[INPUT — first 300 chars]:")
    print(description[:300] + "...")
    print(f"\n[GENERATED BioScript]:")
    print(generated[:600])
    print(f"\n[REFERENCE BioScript — first 300 chars]:")
    print(reference_bs[:300])
    print(f"\n[METRICS]:")
    print(f"  Structural Score : {checks['score']*100:.1f}%")
    print(f"  BLEU-1 Score     : {bleu:.3f}")
    print(f"  Has module       : {checks['has_module']}")
    print(f"  Has manifest     : {checks['has_manifest']}")
    print(f"  Has instructions : {checks['has_instructions']}")
    print(f"  Has dispense     : {checks['has_dispense']}")

del demo_model, demo_tokenizer
torch.cuda.empty_cache()
print(f"\n✅ Inference Demo Complete")

==((====))==  Unsloth 2026.8.19: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROTOCOL 1: lin2019_udpla_single_cell_protein_mrna

[INPUT — first 300 chars]:
PROTOCOL DESCRIPTION

Title        : Ultra-sensitive Digital Quantification of Proteins and mRNA in Single Cells
Source Link  : https://www.nature.com/articles/s41467-019-11531-z
Paper DOI    : https://doi.org/10.1038/s41467-019-11531-z
Journal      : Nature Communications, 2019...

[GENERATED BioScript]:
module fluorescence


platform microscope



// Reagents & Device Layers -------------------------------------------------------

// Bottom layer -- cell culture chamber (~5 uM interparticle spacing; >95% trapping).
cellChamberReagent   = Cell_Culture_Chamber_Storage;
trapAndPurgeReservoir = Trap_and_Purge_Reservoir;

repeat times=2 {
    ccmix = mix(cellChamberReagent with trapAndPurgeReservoir);
}

heat(ccmix at 25c during 5m);

if( cc_mix[success] ) {

    // Top layer -- PCR amplifiable buffer + beads that enter via bottom channels,
    // creating ~5 nL isolated nanoparticle reaction volu

[REFERENCE 

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROTOCOL 2: Multiplex_Newborn_Screening

[INPUT — first 300 chars]:
Source link: https://www.newsteps.org/sites/default/files/resources/download/Multiplex%20Newborn%20Screening%20for%20Pompe%2C%20Fabry%2C%20Hunter.pdf
Paper DOI: https://doi.org/10.1016/j.cca.2013.05.015

Protocol details:

ASSAY TYPE: Digital Microfluidic 5-Plex Fluorometric Enzyme Assay for Newborn...

[GENERATED BioScript]:
module fluorescent_assays

platform dmf_96


// --- REAGENT DECLARATIONS ---

// Standard lab reagents used as well as special DMF-compatible formulations,
// which contain surfactants to prevent droplet coalescence / adsorption during operation.

reagent dbsextract       // single-disk blood spot ELISA sample preparation step;
                        // standard is extracted onto PDMS chip surface then washed away before analysis;

reagent gaa_reagent      //
reagent gla_reagent      //

// Note that glucose-1-phosphate dehydrogenase activity measurements are run simultaneously
// alongside e

[

## 8. Summary & Conclusions

### Key Results

| Model | Struct% | BLEU | Instructions | Best Val Loss |
|-------|---------|------|--------------|---------------|
| Qwen2.5-Coder-3B (3ep) | 67.4% | 0.223 | 6/18 | 1.326 |
| Qwen2.5-Coder-3B (5ep) | 67.4% | 0.261 | 7/18 | 1.330 |
| Llama-3.2-3B (5ep) | 73.6% | 0.344 | 9/18 | 1.418 |
| **Qwen2.5-Coder-7B (5ep)** | **88.2%** | **0.354** | **12/18** | **1.200** |

### Findings

1. **Model size is the strongest predictor of performance** — Qwen-7B 
   outperforms all 3B models by a significant margin (88.2% vs 67.4%)

2. **Code-specialized models outperform general models** — Qwen-7B 
   beats Llama-3B despite comparable parameter counts, validating 
   the use of code-specialized base models for DSL generation

3. **Overfitting is the main challenge** — All models achieve best 
   validation loss at step 50 and overfit beyond that, indicating 
   the dataset size (553 examples) is the primary bottleneck

4. **Epoch count has minimal impact on 3B models** — 3 vs 5 epochs 
   produces identical structural accuracy (67.4%), confirming that 
   more epochs do not help when dataset is small

5. **Zero repetitive outputs** — All models achieve 18/18 
   non-repetitive generation after applying repetition penalty

### Limitations

- Small dataset (174 protocols) limits generalization
- Compiler-based validation not yet integrated
- Single-pass generation without iterative refinement
- Module and variable names differ from reference (expected behavior)

### Future Work

- Expand OpenBioSet with more protocol types
- Integrate BioScript compiler for execution-based evaluation  
- Explore larger models (13B+) with extended context
- Implement RLHF with compiler feedback as reward signal
- Add few-shot prompting for improved name consistency

## 9. Interactive Demo (Gradio)

An interactive web interface for BioGPT — type any biological protocol 
description and get BioScript code instantly using the best model 
(Qwen2.5-Coder-7B-5ep).
The following code launches an interactive Gradio demo for BioGPT.
Run this cell to start the demo locally within the Kaggle session.

**Note:** The public Gradio URL expires after 72 hours.

In [10]:
# ============================================================
# CELL 19 — CODE: Gradio Setup
# ============================================================

import subprocess
subprocess.run(["pip", "install", "-q", "gradio"], check=True)
print("✅ Gradio installed")

✅ Gradio installed


In [12]:
# ============================================================
# CELL 20 — CODE: Gradio App
# ============================================================

import gradio as gr
import torch
from pathlib import Path

# Load best model
print("Loading best model (Qwen2.5-Coder-7B)...")

demo_model, demo_tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(MODELS_DIR / "qwen7b_5ep/lora_adapter"),
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
    token=os.environ["HF_TOKEN"],
)
FastLanguageModel.for_inference(demo_model)
print("✅ Model loaded!")

INFERENCE_PROMPT = (
    "You are BioGPT, an expert biological protocol compiler. "
    "Given a natural language description of a biological laboratory protocol, "
    "generate syntactically correct BioScript (.bs) code.\n\n"
    "You MUST always generate ALL sections in this exact order:\n"
    "1. module declarations\n"
    "2. manifest declarations\n"
    "3. instructions: keyword\n"
    "4. operations (dispense, mix, heat, detect, dispose)\n\n"
    "CRITICAL: Output ONLY valid BioScript code.\n"
    "DO NOT output JSON, XML, or any other format.\n\n"
    "CORRECT FORMAT:\n"
    "module moduleName\n\n"
    "manifest Reagent1\n"
    "manifest Reagent2\n\n"
    "instructions:\n\n"
    "// Step description\n"
    "var1 = dispense Reagent1 into $1 for 5s\n"
    "var2 = dispense Reagent2 into $2 for 5s\n"
    "mix1 = mix var1 with var2 for 30s\n"
    "result = detect fluorescence on mix1 for 10s\n"
)

def biogpt_generate(description, max_tokens=512, temperature=0.7):
    if not description.strip():
        return "Please enter a protocol description.", ""

    chat = [
        {"role": "system", "content": INFERENCE_PROMPT},
        {"role": "user",   "content": description},
    ]
    prompt = demo_tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True,
    )
    prompt += "module "

    inputs = demo_tokenizer(
        prompt, return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN - max_tokens,
    ).to("cuda")

    with torch.no_grad():
        outputs = demo_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.3,
            pad_token_id=demo_tokenizer.eos_token_id,
            eos_token_id=demo_tokenizer.eos_token_id,
        )

    generated = "module " + demo_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()

    checks  = check_structure(generated)
    metrics = (
        f"Structural Score : {checks['score']*100:.1f}%\n"
        f"Has module       : {checks['has_module']}\n"
        f"Has manifest     : {checks['has_manifest']}\n"
        f"Has instructions : {checks['has_instructions']}\n"
        f"Has dispense     : {checks['has_dispense']}\n"
        f"Non-repetitive   : {checks['no_repetition']}"
    )
    return generated, metrics

examples = [
    "This protocol performs PCR amplification. Mix DNA template with primers and PCR master mix, heat to 95C for denaturation, cycle through annealing at 60C and extension at 72C. Detect amplification by fluorescence.",
    "ELISA protocol for protein detection. Dispense sample and antibody, incubate at 37C, wash with PBS buffer, add substrate and detect absorbance signal.",
    "Cell lysis and DNA extraction using digital microfluidics. Dispense cells into lysis buffer, mix thoroughly, heat at 65C to lyse cells, separate DNA using magnetic beads.",
]

with gr.Blocks(title="BioGPT", theme=gr.themes.Soft()) as biogpt_app:
    gr.Markdown("# BioGPT: LLM-Based BioScript Compiler Automation")
    gr.Markdown("**Model:** Qwen2.5-Coder-7B-Instruct fine-tuned on OpenBioSet | **Author:** Ishan Gain")

    with gr.Row():
        with gr.Column(scale=1):
            description_input = gr.Textbox(
                label="Protocol Description",
                placeholder="Describe your biological protocol here...",
                lines=10,
            )
            max_tokens_slider = gr.Slider(
                minimum=128, maximum=1024, value=512, step=64,
                label="Max Output Tokens"
            )
            temperature_slider = gr.Slider(
                minimum=0.1, maximum=1.0, value=0.7, step=0.1,
                label="Temperature"
            )
            generate_btn = gr.Button("Generate BioScript", variant="primary")

        with gr.Column(scale=1):
            bioscript_output = gr.Textbox(
                label="Generated BioScript (.bs)",
                lines=15,
                show_copy_button=True,
            )
            metrics_output = gr.Textbox(
                label="Quality Metrics",
                lines=8,
            )

    generate_btn.click(
        fn=biogpt_generate,
        inputs=[description_input, max_tokens_slider, temperature_slider],
        outputs=[bioscript_output, metrics_output],
    )

    gr.Examples(
        examples=[[ex] for ex in examples],
        inputs=description_input,
        label="Example Protocols"
    )

    gr.Markdown("BioScript compiled by lilott8/BioScript ANTLR4 compiler | Dataset: OpenBioSet (175 protocols)")

biogpt_app.launch(share=True, debug=False)

Loading best model (Qwen2.5-Coder-7B)...
==((====))==  Unsloth 2026.8.19: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✅ Model loaded!


/tmp/ipykernel_58/2850820491.py:96: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="BioGPT", theme=gr.themes.Soft()) as biogpt_app:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c4947b91cf4d60960f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene